In [1]:
import os

In [2]:
start_node = 'AA'

In [3]:
file = 'data/day16.txt'
path = os.path.join(os.getcwd(), file)
with open(path, 'r') as f:
    lines = [x.strip('\n') for x in f.readlines()]

In [4]:
# parsing input into a crude data structure
valves = {}
for row in lines:
    row = row.replace('Valve ', '').replace(' has flow rate=', ', ')
    row = row.replace('; tunnels lead to valves', ',')
    row = row.replace('; tunnel leads to valve', ',')
    row = row.split(', ')
    valves[row[0]] = row[1:]
flow_valves = [start_node] + [x for x in valves.keys() if valves[x][0] != '0']
path_lengths = {x: [] for x in flow_valves}
for valve in flow_valves:
    crawl, path, exclude_list, count = valves[valve][1:], [], [valve], 0
    flow_rate = valves[valve][0]
    while True:
        count += 1
        next_crawl = []
        for elem in crawl:
            if elem not in exclude_list:
                next_crawl.extend(valves[elem][1:])
                if elem in flow_valves:
                    path.append((elem, count))
            exclude_list.append(elem)
        if len(next_crawl) == 0:
            path_lengths[valve] = [flow_rate] + path
            break
        crawl = next_crawl

In [5]:
# part one
time_limit = 30
start_time, start_flow_rate, start_flow, max_flow = 0, 0, 0, [0]
paths = [[start_time, start_flow_rate, start_flow, start_node]]
max_rate = sum([int(path_lengths[key][0]) for key in path_lengths])
path_check = set(flow_valves)
while paths:
    new_paths = []
    for path in paths:
        time, flow_rate, flow = path[0], path[1], path[2]
        # this if block is required to solve for the test case
        flow_remaining = max_rate * (time_limit - time)
        if path_check == set(path[3:]):
            flow += flow_remaining
            max_flow.append(flow)
            continue
        nodes = path_lengths[path[-1]][1:]
        for node in nodes:
            valve, node_time = node
            if valve not in path:
                new_time = time + node_time + 1
                if new_time >= time_limit:
                    new_flow = flow + (flow_rate * (time_limit - time))
                    max_flow.append(new_flow)
                else:
                    new_flow = flow + (flow_rate * (node_time + 1))
                    new_rate = flow_rate + int(path_lengths[valve][0])
                    pre_path = [new_time, new_rate, new_flow]
                    new_path = pre_path + path[3:] + [valve]
                    new_paths.append(new_path)
    paths = new_paths
print(max(max_flow))

2119


In [6]:
# part two
max_time = 26
start_time, start_rate, start_flow, max_flow = 0, 0, 0, [0]
h_path = [start_time, start_rate, start_flow, start_node]
e_path = [start_time, start_rate, start_flow, start_node]
paths = [[h_path, e_path]]
while paths:
    new_paths, current_max = [], max(max_flow)
    print('paths processing', len(paths))
    for path in paths:
        h_path, e_path = path[0], path[1]
        exclude_list = h_path + e_path
        h_time, h_rate, h_flow = h_path[0], h_path[1], h_path[2]
        e_time, e_rate, e_flow = e_path[0], e_path[1], e_path[2]
        h_time_rem = max_time - h_time
        e_time_rem = max_time - e_time
        if path_check == set(h_path[3:] + e_path[3:]):
            print(h_path,e_path)
            h_flow_rem = h_time_rem * h_rate
            e_flow_rem = e_time_rem * e_rate
            final_flow = h_flow_rem + e_flow_rem + h_flow + e_flow
            max_flow.append(final_flow)
            continue
        h_nodes = path_lengths[h_path[-1]][1:]
        e_nodes = path_lengths[e_path[-1]][1:]
        for h_node in h_nodes:
            h_valve, h_node_time = h_node
            h_node_time += 1
            if h_valve not in exclude_list:
                for e_node in e_nodes:
                    e_valve, e_node_time = e_node
                    e_node_time += 1
                    if e_valve not in exclude_list and h_valve != e_valve:
                        h_new_rate = h_rate + int(path_lengths[h_valve][0])
                        h_new_time = h_time + h_node_time
                        if h_new_time >= max_time:
                            h_new_time = max_time
                            h_new_flow = h_flow + (h_rate * max_time - h_time)
                        else:
                            h_new_flow = h_flow + (h_rate * h_node_time)
                        pre_path = [h_new_time, h_new_rate, h_new_flow]
                        h_new_path = pre_path + h_path[3:] + [h_valve]
                        e_new_rate = e_rate + int(path_lengths[e_valve][0])
                        e_new_time = e_time + e_node_time
                        if e_new_time >= max_time:
                            e_new_time = max_time
                            e_new_flow = e_flow + (e_rate * max_time - e_time)
                        else:
                            e_new_flow = e_flow + (e_rate * e_node_time)
                        pre_path = [e_new_time, e_new_rate, e_new_flow]
                        e_new_path = pre_path + e_path[3:] + [e_valve]
                        if h_new_time == max_time and e_new_time == max_time:
                            max_flow.append(h_new_flow + e_new_flow)
                        else:
                            current_flow = h_flow + e_flow
                            pot_flow = max_rate * (h_time_rem + e_time_rem)
                            if current_flow + pot_flow > 2320:
                                new_paths.append([h_new_path, e_new_path])
                    else:
                        continue
    paths = new_paths
print(max(max_flow))

paths processing 1
paths processing 210
paths processing 32748
paths processing 3453196
